In [20]:
import requests
import re
import pandas as pd
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [21]:
# Endereço da página que contém a lista de plantas
URL_CATALOGO = "https://www.pancdocerrado.com.br/catalogo.html"
# Endereço-base do site
URL_BASE = "https://www.pancdocerrado.com.br/"

# Criar uma sessão para realizar as requisições
sessao = requests.Session()

# Identificar a requisição como proveniente de um navegador
sessao.headers.update({
    "User-Agent": "Mozilla/5.0"
})

In [22]:
# Realizar uma requisição HTTP para acessar o catálogo
resposta = sessao.get(URL_CATALOGO, timeout=20)

# Verificar se a requisição foi realizada corretamente
resposta.raise_for_status()

# Interpretar o conteúdo HTML da página
soup = BeautifulSoup(resposta.text, "html.parser")

In [23]:
# Lista que armazenará o nome e a URL de cada planta
lista_plantas = []

# Percorrer todos os elementos script da página
for script in soup.find_all("script"):

    # Obter o conteúdo JavaScript
    conteudo = script.string or script.get_text()

    # Procurar o array que começa com const plantas = [...]
    resultado = re.search(
        r"const\s+plantas\s*=\s*\[(.*?)\]\s*;",
        conteudo,
        re.DOTALL
    )

    # Verificar se o array foi encontrado
    if resultado:

        # Extrair o conteúdo interno do array
        array_plantas = resultado.group(1)

        # Extrair o nome e a URL de cada planta
        lista_plantas = re.findall(
            r'\{\s*nome:\s*"([^"]+)"\s*,\s*url:\s*"([^"]+)"\s*\}',
            array_plantas
        )

        # Interromper a busca após encontrar a lista
        break


# Exibir a quantidade de plantas encontradas
print(f"Total de plantas encontradas: {len(lista_plantas)}")

# Visualizar os primeiros registros
print(lista_plantas[:5])

Total de plantas encontradas: 63
[('Açafrão-da-terra : Curcuma longa L.', 'acafrao-da-terra.html'), ('Alho-nirá : Allium tuberosum Rottler ex Spreng', 'alho-nira.html'), ('Almeirão-roxo : Lactuca canadensis L', 'almeirao-roxo.html'), ('Araçá : Psidium cattleyanum Sabine', 'araca.html'), ('Araticum : Annona crassiflora Mart.', 'araticum.html')]


In [24]:
# Lista que armazenará os dados extraídos de cada planta
dados_plantas = []

# Percorrer todas as plantas encontradas no catálogo
for nome, caminho_url in lista_plantas:

    # Construir a URL completa da página da planta
    url_planta = urljoin(URL_BASE, caminho_url)

    # Exibir o nome da planta que está sendo processada
    print(f"Extraindo: {nome}")

    try:

        # Acessar a página individual da planta
        resposta_planta = sessao.get(url_planta, timeout=20)

        # Verificar se a requisição foi realizada corretamente
        resposta_planta.raise_for_status()

        # Interpretar o HTML da página
        pagina = BeautifulSoup(
            resposta_planta.text,
            "html.parser"
        )

    except requests.RequestException as erro:

        # Exibir o erro caso não seja possível acessar a página
        print(f"Erro ao acessar {url_planta}: {erro}")

        continue

    # Aguardar meio segundo antes da próxima requisição
    time.sleep(0.5)

Extraindo: Açafrão-da-terra : Curcuma longa L.
Extraindo: Alho-nirá : Allium tuberosum Rottler ex Spreng
Extraindo: Almeirão-roxo : Lactuca canadensis L
Extraindo: Araçá : Psidium cattleyanum Sabine
Extraindo: Araticum : Annona crassiflora Mart.
Extraindo: Araruta : Maranta arundinacea L
Extraindo: Aroeira : Schinus terebinthifolia Raddi
Extraindo: Azedinha : Rumex acetosa L.
Extraindo: Bacupari : Garcinia gardneriana (Planch. & Triana) Zappi
Extraindo: Banha de Galinha : Swartzia multijuga Vogel
Extraindo: Baru : Dipteryx alata Vogel 
Extraindo: Beldroega : Dipteryx alata Vogel 
Extraindo: Bertalha : Basella alba L
Extraindo: Biribiri : Averrhoa bilimbi L.
Extraindo: Buriti : Mauritiella armata (Mart.) Burret.
Extraindo: Buva : Conyza bonariensis (L.) Cronquist
Extraindo: Cagaita : Eugenia dysenterica DC
Extraindo: Cajazinho : Spondias mombin L.
Extraindo: Cajuzinho-do-cerrado : Anacardium humile A. St.-Hil.
Extraindo: Calabura : Muntingia calabura L.
Extraindo: Capim-limão : Cymbopog

In [25]:
# Percorrer novamente as plantas para extrair suas informações
for nome, caminho_url in lista_plantas:

    url_planta = urljoin(URL_BASE, caminho_url)

    try:

        resposta_planta = sessao.get(url_planta, timeout=20)
        resposta_planta.raise_for_status()

        pagina = BeautifulSoup(
            resposta_planta.text,
            "html.parser"
        )

        # Criar um dicionário para armazenar os dados da planta
        planta = {
            "Nome da planta": nome,
            "URL": url_planta
        }

        # Localizar os elementos que contêm as informações técnicas
        informacoes = pagina.find_all(
            "div",
            class_="course-info"
        )

        # Percorrer cada informação técnica encontrada
        for info in informacoes:

            # Localizar o título e o conteúdo da informação
            titulo = info.find("h5")
            valor = info.find("p")

            if titulo and valor:

                # Extrair o título e remover os dois-pontos
                chave = titulo.get_text(
                    strip=True
                ).replace(":", "")

                # Extrair o conteúdo textual
                conteudo = valor.get_text(
                    " ",
                    strip=True
                )

                # Armazenar apenas informações com conteúdo
                if conteudo:
                    planta[chave] = conteudo

        # Adicionar o dicionário à lista de plantas
        dados_plantas.append(planta)

    except requests.RequestException as erro:

        print(f"Erro ao acessar {url_planta}: {erro}")

    # Aguardar antes de acessar a próxima página
    time.sleep(0.5)

In [32]:
# Transformar a lista de dicionários em um DataFrame
df = pd.DataFrame(dados_plantas)

# Exibir as primeiras cinco plantas
display(df.head())

,Nome da planta,URL,Veja em 3D,Nome Popular,Nome Científico,Família,Sazonalidade,Sabor,Sugestão de Preparo,Função alimentar,...,Distribuição Geográfica - Ocorrências Confirmadas,Domínios Fitogeográficos,Tipo de Vegetação,Cultivo,Descrição,Possíveis ocorrências,Distribuição Geográfica - Possíveis ocorrências,Distribuição Geográfica - Possíveis Ocorrências,Ilhas Oceânicas - Ocorrências Confirmadas,Distribuição Geográfica - Possívies Ocorrências
0,Açafrão-da-terra : Curcuma longa L.,https://www.pancdocerrado.com.br/acafrao-da-te...,Fruto,"Açafrão-da-terra, Cúrcuma, Açafroa, Açafrão-da...",Curcuma longa L.,Zingiberaceae,Junho-Agosto,"Terroso, Levemente Amargo e Picante",Pipoca com Açafrão,"Rizoma com propriedades anti-inflamatórias, ri...",...,Nordeste (Piauí),"Cerrado, Mata Atlântica e Amazônia",Área Antrópica,Rizomas,"Caule: herbácea rizomatosa, perere, ereta, ces...",NaN,NaN,NaN,NaN,NaN
1,Alho-nirá : Allium tuberosum Rottler ex Spreng,https://www.pancdocerrado.com.br/alho-nira.html,Folha,"Alho-nirá, Cebolinha-achatada, Cebolinha-chata...",Allium tuberosum Rottler ex Spreng,Amaryllidaceae,Janeiro-Dezembro,"Combinação de alho, alho-poró e cebolinha, por...",Patê Verde com Folhas de Alho Nirá,"Folhas com sabor semelhante ao alho, ricas em ...",...,"Sul (Santa Catarina, Rio Grande do Sul); Sudes...","(Amazônia, Caatinga, Cerrado, Mata Atlântica, ...","Floresta Ombrófila Mista, Floresta Atlântica, ...",Semente ou Touceira,"Caule: herbácea rizomatosa, perene, acaule, er...",NaN,NaN,NaN,NaN,NaN
2,Almeirão-roxo : Lactuca canadensis L,https://www.pancdocerrado.com.br/almeirao-roxo...,Folha,"Almeirão-roxo, Almeirão-de-árvore, Almeirão-ja...",Lactuca canadensis L,Asteraceae,Janeiro-Dezembro,Levemente Amargo e Refrescante,Salada de Almeirão-Roxo,Folhas amargas ricas em fibras e antioxidantes.,...,Sudeste (São Paulo); Sul (Paraná),Mata Atlântica,Área Antrópica,Semente e Touceira,Caule: cor do látex laranja/amarelado.\n ...,NaN,NaN,NaN,NaN,NaN
3,Araçá : Psidium cattleyanum Sabine,https://www.pancdocerrado.com.br/araca.html,Fruto,"Araçá, Araçá-comum, Araçá-do-campo, Araçá-da-p...",Psidium cattleyanum Sabine,Myrtaceae,Novembro-Fevereiro,Adocicado com Acidez Equilibrada,Doce de “bolinha de Araçá”,Fruto rico em vitamina C e compostos antioxida...,...,"Nordeste (Alagoas, Bahia, Ceará, Pernambuco, S...","Caatinga, Cerrado, Mata Atlântica","Cerrado (lato sensu), Floresta Ombrófila (Flor...","Semente, Estaquia e Alporquia",Caule: alt. (m) médio das planta(s) entre 3 e ...,NaN,NaN,NaN,NaN,NaN
4,Araticum : Annona crassiflora Mart.,https://www.pancdocerrado.com.br/araticum.html,Fruto,"Araticum, Araticum do Cerrado, Acanga, Bruto, ...",Annona crassiflora Mart.,Annonaceae,Janeiro-Abril,Doce,Mousse de Araticum,Polpa nutritiva rica em carboidratos e minerais.,...,"Norte (Pará, Rondônia, Roraima, Tocantins); No...","Amazônia, Caatinga, Cerrado, Pantanal","Campo Rupestre, Cerrado (lato sensu)",Semente,"Árvore ou arbusto 3–18 m alt.; ramos, gemas e ...",NaN,NaN,NaN,NaN,NaN


In [33]:
df = df.drop(columns=[
    "Veja em 3D",
    "Sabor",
    "Sugestão de Preparo",
    "Função alimentar",
    "Função ecológica",
    "Origem",
    "Endemismo",
    "Tipo de Vegetação",
    "Cultivo" ,
    "Possíveis ocorrências",
    "Distribuição Geográfica - Possíveis ocorrências",
    "Distribuição Geográfica - Possíveis Ocorrências",
    "Ilhas Oceânicas - Ocorrências Confirmadas",
    "Distribuição Geográfica - Possívies Ocorrências",
    "Forma de Vida",
    "Família",
    "Substrato"
] )

In [34]:
df

,Nome da planta,URL,Nome Popular,Nome Científico,Sazonalidade,Distribuição Geográfica - Ocorrências Confirmadas,Domínios Fitogeográficos,Descrição
0,Açafrão-da-terra : Curcuma longa L.,https://www.pancdocerrado.com.br/acafrao-da-te...,"Açafrão-da-terra, Cúrcuma, Açafroa, Açafrão-da...",Curcuma longa L.,Junho-Agosto,Nordeste (Piauí),"Cerrado, Mata Atlântica e Amazônia","Caule: herbácea rizomatosa, perere, ereta, ces..."
1,Alho-nirá : Allium tuberosum Rottler ex Spreng,https://www.pancdocerrado.com.br/alho-nira.html,"Alho-nirá, Cebolinha-achatada, Cebolinha-chata...",Allium tuberosum Rottler ex Spreng,Janeiro-Dezembro,"Sul (Santa Catarina, Rio Grande do Sul); Sudes...","(Amazônia, Caatinga, Cerrado, Mata Atlântica, ...","Caule: herbácea rizomatosa, perene, acaule, er..."
2,Almeirão-roxo : Lactuca canadensis L,https://www.pancdocerrado.com.br/almeirao-roxo...,"Almeirão-roxo, Almeirão-de-árvore, Almeirão-ja...",Lactuca canadensis L,Janeiro-Dezembro,Sudeste (São Paulo); Sul (Paraná),Mata Atlântica,Caule: cor do látex laranja/amarelado.\n ...
3,Araçá : Psidium cattleyanum Sabine,https://www.pancdocerrado.com.br/araca.html,"Araçá, Araçá-comum, Araçá-do-campo, Araçá-da-p...",Psidium cattleyanum Sabine,Novembro-Fevereiro,"Nordeste (Alagoas, Bahia, Ceará, Pernambuco, S...","Caatinga, Cerrado, Mata Atlântica",Caule: alt. (m) médio das planta(s) entre 3 e ...
4,Araticum : Annona crassiflora Mart.,https://www.pancdocerrado.com.br/araticum.html,"Araticum, Araticum do Cerrado, Acanga, Bruto, ...",Annona crassiflora Mart.,Janeiro-Abril,"Norte (Pará, Rondônia, Roraima, Tocantins); No...","Amazônia, Caatinga, Cerrado, Pantanal","Árvore ou arbusto 3–18 m alt.; ramos, gemas e ..."
...,...,...,...,...,...,...,...,...
58,Serralha : Sonchus oleraceus L.,https://www.pancdocerrado.com.br/serralha.html,"Serralha, Chicória-brava, Chicória-do-mato, Se...",Sonchus oleraceus L.,Janeiro-Dezembro,Sul (Rio Grande do Sul),"Caatinga, Cerrado, Mata Atlântica",Caule: planta(s) de caule(s) herbáceo(s) glabr...
59,Taioba : Xanthosoma taioba,https://www.pancdocerrado.com.br/taioba.html,"Taioba, Orelha de Elefante, Mangarás, Macabo, ...",Xanthosoma taioba,Janeiro-Dezembro,"Norte (Acre, Amazonas, Pará, Rondônia); Nordes...","Amazônia, Cerrado, Mata Atlântica",Raiz: adventícia(s) não aplicável. Caule: rizo...
60,Tanchagem : Plantago major L.,https://www.pancdocerrado.com.br/tanchagem.html,"Tanchagem, Tanchagem-grande, Tansagem, Plantag...",Plantago major L.,Janeiro-Dezembro,"Norte (Acre, Pará);\n Nordeste ...","Floresta Amazônica, Caatinga, Cerrado, Mata At...",Raiz: raiz(es) axial(ais) presente(s) não engr...
61,Trapoeraba : Commelina benghalensis L,https://www.pancdocerrado.com.br/trapoeraba.html,"Trapoeraba, Trapoeraba-azul, Andacá, Trapoerab...",Commelina benghalensis L,Janeiro-Dezembro,"Norte (Acre, Amazonas, Amapá, Pará, Rondônia, ...","Amazônia, Caatinga, Cerrado, Mata Atlântica, P...",Caule: posição ereto(s)-decumbente(s).\n ...


In [37]:
df.to_csv("plantas_panc_cerrado.csv", index=False, encoding="utf-8-sig")
print(f"Plantas extraídas: {len(dados_plantas)}")

Plantas extraídas: 63
